# Week 14: LLM Fine-Tuning & Quantization
## Notebook 12: Model Context Protocol (MCP): Open Tool Interoperability

### 1. Conceptual Intuition & Theoretical Background
Early AI agent architectures required custom, bespoke integration code for every external tool, database, and API.

**Model Context Protocol (MCP, Anthropic, 2024):**
Establishes an open, standardized protocol decoupling LLMs from external tools and data sources:
- **Client-Server Architecture:** The AI host application (Client) connects to external services (Servers) via standardized JSON-RPC 2.0 messages over standard I/O (`stdio`) or Server-Sent Events (SSE).
- **Core Primitives:**
  1. **Tools:** Executable functions callable by the LLM (e.g., query database, execute shell command).
  2. **Resources:** Read-only data streams (e.g., file contents, system logs).
  3. **Prompts:** Pre-engineered contextual workflow templates.

---

### 2. Scratch Python Implementation: MCP JSON-RPC Server & Client

In [ ]:
import json

class MockMCPServer:
    """Simulates an MCP Server exposing Tool and Resource primitives."""
    def __init__(self):
        self.tools = {
            "get_stock_quote": {
                "description": "Retrieves real-time stock price given a ticker symbol.",
                "parameters": {
                    "type": "object",
                    "properties": {"ticker": {"type": "string"}},
                    "required": ["ticker"]
                }
            }
        }
        
    def handle_request(self, json_rpc_str):
        req = json.loads(json_rpc_str)
        method = req.get("method")
        msg_id = req.get("id")
        
        if method == "tools/list":
            return json.dumps({
                "jsonrpc": "2.0",
                "id": msg_id,
                "result": {"tools": self.tools}
            })
            
        elif method == "tools/call":
            params = req.get("params", {})
            tool_name = params.get("name")
            args = params.get("arguments", {})
            
            if tool_name == "get_stock_quote":
                ticker = args.get("ticker", "UNKNOWN")
                return json.dumps({
                    "jsonrpc": "2.0",
                    "id": msg_id,
                    "result": {"content": [{"type": "text", "text": f"Price of {ticker}: $185.20 USD"}]}
                })
                
        return json.dumps({"jsonrpc": "2.0", "id": msg_id, "error": {"code": -32601, "message": "Method not found"}})

# Simulate Client Interacting via Standardized JSON-RPC
server = MockMCPServer()

# 1. Discovery Phase
list_request = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list"})
list_response = server.handle_request(list_request)

# 2. Execution Phase
call_request = json.dumps({
    "jsonrpc": "2.0",
    "id": 2,
    "method": "tools/call",
    "params": {"name": "get_stock_quote", "arguments": {"ticker": "AAPL"}}
})
call_response = server.handle_request(call_request)

print("--- Model Context Protocol (MCP) Protocol Exchanges ---")
print("1. Tool Discovery Response:")
print(json.dumps(json.loads(list_response), indent=2))
print("\n2. Tool Execution Response:")
print(json.dumps(json.loads(call_response), indent=2))

### 3. Step-by-Step Code Walkthrough
1. **Discovery Handshake (`tools/list`):** Discovers capabilities dynamically without hardcoding tool schemas.
2. **Structured Execution (`tools/call`):** Transports arguments and results via standardized JSON-RPC 2.0 envelopes.
3. **Decoupled Architecture:** Servers can run in isolated Docker containers, remote machines, or local sub-processes.

---

### 4. Industry Applications & Trade-offs
- **Modern AI Development IDEs (Claude Desktop, Cursor):** Integrating tools across GitHub, Postgres, and web scrapers via a unified protocol.
- **Trade-off:** JSON serialization adds a minor latency overhead relative to native Python function calls.

---

### 5. Top Interview Questions & Answers
**Q1: What problem does the Model Context Protocol (MCP) solve for agent development?**
> **Answer:** It establishes a standardized client-server protocol for tool and resource integration, avoiding the $M \times N$ integration problem where every LLM framework needs custom adapters for every third-party service.

**Q2: What is the difference between an MCP Tool and an MCP Resource?**
> **Answer:** A Resource is a passive data endpoint that provides read-only context (e.g. document contents). A Tool is an active executable function with potential side effects (e.g. running a query, creating a file).